In [ ]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_REV = '4a5963f'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-data-kaggle.txt


In [ ]:
from spider.exp4_data import find_exp4_data, restore_action_evaluation_shards

prepared = find_exp4_data('/kaggle/input')
os.environ['SPIDER_DATA_DIR'] = str(prepared)
labels = ['action-base-shard-00-of-02', 'action-base-shard-01-of-02', 'action-exp002-shard-00-of-02', 'action-exp002-shard-01-of-02']
restored = restore_action_evaluation_shards('/kaggle/input', labels, REPO_ROOT)
print({'event': 'action_shards_restored', 'paths': [str(p) for p in restored]})


In [ ]:
from spider.action_merge import merge_action_shards

_, base_metrics = merge_action_shards('configs/experiment4.yaml', 'action-base', ['action-base-shard-00-of-02', 'action-base-shard-01-of-02'], 'development')
_, exp2_metrics = merge_action_shards('configs/experiment4.yaml', 'action-exp002', ['action-exp002-shard-00-of-02', 'action-exp002-shard-01-of-02'], 'development')
print({'event': 'action_baselines_merged', 'base': base_metrics, 'exp002': exp2_metrics}, flush=True)
